In [2]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.8 MB/s eta 0:00:00


In [9]:
import pandas as pd
from surprise import Dataset
from surprise import SVD, SVDpp, NMF
from surprise.model_selection import GridSearchCV
from surprise.model_selection import cross_validate

In [4]:
data = Dataset.load_builtin("ml-100k")

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


In [5]:
param_grid = {
    "n_factors": [50, 100],
    "n_epochs": [20, 30],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.1]
}
gs = GridSearchCV(SVD,param_grid,measures=["rmse"],cv=3)
gs.fit(data)
print("Найкращий RMSE:")
print(gs.best_score["rmse"])
print("\nНайкращі параметри:")
print(gs.best_params["rmse"])

Найкращий RMSE:
0.93516849900177

Найкращі параметри:
{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


In [6]:
best_svd = SVD(**gs.best_params["rmse"])

In [7]:
svd_result = cross_validate(best_svd,data,measures=["RMSE", "MAE"],cv=5,verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9313  0.9268  0.9206  0.9261  0.9264  0.9262  0.0034  
MAE (testset)     0.7371  0.7351  0.7293  0.7344  0.7332  0.7338  0.0026  
Fit time          1.81    3.21    2.35    2.40    1.58    2.27    0.57    
Test time         0.10    0.36    0.12    0.22    0.13    0.19    0.10    


In [8]:
svdpp = SVDpp()
svdpp_result = cross_validate(svdpp,data,measures=["RMSE", "MAE"],cv=5,verbose=True)

Evaluating RMSE, MAE of algorithm SVDpp on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9120  0.9272  0.9196  0.9227  0.9207  0.9204  0.0050  
MAE (testset)     0.7147  0.7259  0.7220  0.7236  0.7235  0.7220  0.0038  
Fit time          25.63   22.45   26.95   26.02   25.61   25.33   1.52    
Test time         5.79    5.14    4.32    5.60    6.50    5.47    0.72    


In [10]:
nmf = NMF()
nmf_result = cross_validate(nmf,data,measures=["RMSE", "MAE"],cv=5,verbose=True)

Evaluating RMSE, MAE of algorithm NMF on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9646  0.9597  0.9563  0.9724  0.9587  0.9623  0.0057  
MAE (testset)     0.7569  0.7544  0.7500  0.7637  0.7558  0.7561  0.0044  
Fit time          2.22    3.23    3.29    2.46    2.77    2.79    0.42    
Test time         0.09    0.23    0.23    0.10    0.32    0.19    0.09    


In [11]:
results = pd.DataFrame({
    "Model": ["SVD", "SVD++", "NMF"],
    "RMSE": [svd_result["test_rmse"].mean(),svdpp_result["test_rmse"].mean(),nmf_result["test_rmse"].mean()],
    "MAE": [svd_result["test_mae"].mean(),svdpp_result["test_mae"].mean(),nmf_result["test_mae"].mean()]})
results

,Model,RMSE,MAE
0,SVD,0.926230,0.733815
1,SVD++,0.920439,0.721956
2,NMF,0.962326,0.756145
